# Ember Malware Detection: Native Multimodal Research Baseline (Methodologically Corrected)

**Author:** Arjun Singh Rajput (GLA University / IIIT Kottayam)  
**Project:** Ember – Malware Detection Using Native Multimodal Learning  
**Git Branch:** `experiment/native-multimodal-transformer`  
**Dataset:** EMBER 2018 v2 (JSONL feature records)  
**Environment:** Google Colab (GPU-accelerated / CPU-adaptive)  
**Protocol:** Methodological Baseline Correction (Reservoir Sampling & Modern Scikit-Learn Hash Parity)  

--- 

### Methodological Audit & Experimental Corrections:
1. **Full-Stream Reservoir Sampling (Algorithm R):** Replaced early-break quota sampling with seeded full-stream reservoir sampling across all 800,000 records in shards 0–5. Every benign and malware record across shards has identical inclusion probability, eliminating file-ordering selection bias while maintaining exact proportional shard representations.
2. **LIEF & Modern Scikit-Learn Compatibility:** Pre-extracted JSONL parsing does not call LIEF at runtime. Monkey-patched `ember.features.SectionInfo.process_raw_features` using `[list(entry_val)]` to ensure 100% mathematical equivalency with scikit-learn 0.20's MurmurHash3 character hashing, verified with zero feature splits in `ember_model_2018.txt` and max prediction difference < 0.000025.
3. **Five Fair Controlled Baselines:**
   - **Tabular Baseline:** Fresh LightGBM (1,000 trees, lr=0.05, num_leaves=31) on official 2,381 EMBER features.
   - **Experiment A:** Byte-Statistics MLP (`ByteMLP`, 512-D).
   - **Experiment B:** Textual Metadata CNN (`MaskedTextCNN`, train-only vocab, padding-safe masked average pooling).
   - **Experiment C:** Structural PE MLP (`StructMLP`, 66-D, StandardScaler fitted on train only).
   - **Experiment A+B+C:** Three-Modality Multimodal Fusion (`MultimodalFusionNet`, joint 320-D representation).
4. **Quarantine:** Test set (`test_features.jsonl`) remains 100% quarantined.

## 1. Environment Setup, Dependency Installation & Hardware Audit

In [ ]:
# Kernel environment verification
import sys
if sys.version_info >= (3, 13):
    print('=' * 75)
    print('[WARNING] ACTIVE NOTEBOOK KERNEL MISMATCH DETECTED!')
    print(f'Current kernel is running: Python {sys.version.split()[0]} from:')
    print(f'  {sys.executable}')
    print('All ML packages (PyTorch, LightGBM, EMBER, NumPy) are installed in:')
    print(r'  Python 3.12.10 (C:\Users\imsto\AppData\Local\Programs\Python\Python312\python.exe)')
    print('\nACTION IN VS CODE:')
    print('  1. Click "Select Kernel" in the top-right corner of the editor.')
    print('  2. Click "Python Environments..." and select "Python 3.12.10".')
    print('=' * 75)

# Install required dependencies (using %pip targets the active kernel)
%pip install -q lief lightgbm torch torchvision scikit-learn scipy tqdm matplotlib pyyaml
%pip install -q git+https://github.com/endgameinc/ember.git

import os
import json
import time
import math
import random
from collections import Counter
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction import FeatureHasher
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
import lief
import lightgbm as lgb
import ember

# Patch EMBER SectionInfo for modern scikit-learn (>=1.0) compatibility
_orig_section_process = ember.features.SectionInfo.process_raw_features
def _patched_section_process(self, raw_obj):
    sections = raw_obj.get('sections', [])
    general = [
        len(sections),
        sum(1 for s in sections if s.get('size', 0) == 0),
        sum(1 for s in sections if s.get('name', '') == ''),
        sum(1 for s in sections if 'MEM_READ' in s.get('props', []) and 'MEM_EXECUTE' in s.get('props', [])),
        sum(1 for s in sections if 'MEM_WRITE' in s.get('props', []))
    ]
    section_sizes = [(s.get('name', ''), s.get('size', 0)) for s in sections]
    section_sizes_hashed = FeatureHasher(50, input_type='pair').transform([section_sizes]).toarray()[0]
    section_entropy = [(s.get('name', ''), s.get('entropy', 0)) for s in sections]
    section_entropy_hashed = FeatureHasher(50, input_type='pair').transform([section_entropy]).toarray()[0]
    section_vsize = [(s.get('name', ''), s.get('vsize', 0)) for s in sections]
    section_vsize_hashed = FeatureHasher(50, input_type='pair').transform([section_vsize]).toarray()[0]
    entry_val = raw_obj.get('entry', '')
    entry_name_hashed = FeatureHasher(50, input_type='string').transform([list(entry_val)]).toarray()[0]
    characteristics = [p for s in sections for p in s.get('props', []) if s.get('name', '') == entry_val]
    characteristics_hashed = FeatureHasher(50, input_type='string').transform([characteristics]).toarray()[0]

    return np.hstack([
        general, section_sizes_hashed, section_entropy_hashed, section_vsize_hashed, entry_name_hashed,
        characteristics_hashed
    ]).astype(np.float32)

ember.features.SectionInfo.process_raw_features = _patched_section_process

# Environment & Hardware Audit
print('=' * 70)
print('GOOGLE COLAB / RUNTIME & HARDWARE AUDIT')
print('=' * 70)
print(f'Python Version      : {sys.version.split()[0]}')
print(f'PyTorch Version     : {torch.__version__}')
print(f'CUDA Available      : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Hardware Model  : {gpu_name}')
    print(f'GPU Total VRAM      : {gpu_vram:.2f} GB')
    BATCH_SIZE = 64
else:
    device = torch.device('cpu')
    gpu_name = 'CPU'
    gpu_vram = 0.0
    print('GPU Hardware Model  : None (CPU Runtime)')
    BATCH_SIZE = 32

print(f'LightGBM Version    : {lgb.__version__}')
print(f'LIEF Version        : {lief.__version__}')
print(f'NumPy Version       : {np.__version__}')
print(f'Pandas Version      : {pd.__version__}')
print(f'Scikit-Learn Version: {sklearn.__version__}')
print(f'Adapted Batch Size  : {BATCH_SIZE}')
print('=' * 70)

# Set Fixed Random Seed for 100% Deterministic Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print('[OK] Random seeds locked to 42 across all frameworks.')

## 2. Environment Path Configuration & Workspace Setup (Colab & Local Compatible)

In [ ]:
# Auto-detect Google Colab vs Local Jupyter Notebook
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/Final Year Paper/ember_dataset_2018_2/ember2018'
    EXP_ROOT_DIR = '/content/drive/MyDrive/EMBER_Experiments/baseline_correction_2026_10_30'
    print('[INFO] Executing in Google Colab environment. Drive mounted.')
else:
    # Local paths (Windows local workspace)
    DATA_DIR = r'c:\Users\imsto\Desktop\Final Year Paper\ember_dataset_2018_2\ember2018'
    EXP_ROOT_DIR = r'c:\Users\imsto\Desktop\Final Year Paper\Old Repo Data\experiments\baseline_correction_2026_10_30'
    print(f'[INFO] Executing in Local Jupyter/Windows environment.')

# Dedicated Experiment Output Directory Structure
METRICS_DIR = os.path.join(EXP_ROOT_DIR, 'metrics')
CHECKPOINTS_DIR = os.path.join(EXP_ROOT_DIR, 'checkpoints')
PLOTS_DIR = os.path.join(EXP_ROOT_DIR, 'plots')
LOGS_DIR = os.path.join(EXP_ROOT_DIR, 'logs')
CONFIGS_DIR = os.path.join(EXP_ROOT_DIR, 'configs')
DATA_CACHE_DIR = os.path.join(EXP_ROOT_DIR, 'data')

for d in [METRICS_DIR, CHECKPOINTS_DIR, PLOTS_DIR, LOGS_DIR, CONFIGS_DIR, DATA_CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

assert os.path.exists(DATA_DIR), f'[ERROR] Dataset directory not found: {DATA_DIR}'
print(f'[OK] Verified dataset directory: {DATA_DIR}')

# Verify the 6 training shard files exist
for i in range(6):
    p = os.path.join(DATA_DIR, f'train_features_{i}.jsonl')
    assert os.path.exists(p), f'Missing shard file: {p}'
    print(f'  [OK] Shard train_features_{i}.jsonl ({os.path.getsize(p)/1024/1024:.1f} MB)')

print('\n' + '=' * 70)
print('TEST SET INTEGRITY GUARANTEE:')
print('The official chronological test set (test_features.jsonl) is strictly')
print('QUARANTINED. It is NEVER loaded, inspected, or evaluated.')
print('=' * 70)

## 3. Native Multimodal Feature Extraction & Train-Only Vocabulary Builder
- **Modality A:** 512-D Byte Histogram & Byte-Entropy (Sum-Normalized Probabilities)
- **Modality B:** API Imports, Exports & Section Names (Configurable Sequence Length)
- **Modality C:** 66-D Structural PE Numerical Metrics (Log1p transformed counters)

In [ ]:
# Configurable sequence length and vocabulary size
MAX_SEQ_LEN = 256  # Configurable: 128 (64.4%), 256 (83.9%), 384 (91.0%), 512 (94.7%)
VOCAB_SIZE = 10000

def extract_modality_a(record):
    """Modality A: 512-D Byte-Level Statistics."""
    hist = np.array(record.get('histogram', [0]*256), dtype=np.float32)
    h_sum = hist.sum()
    hist_norm = hist / (h_sum + 1e-7) if h_sum > 0 else hist
    
    ent = np.array(record.get('byteentropy', [0]*256), dtype=np.float32)
    e_sum = ent.sum()
    ent_norm = ent / (e_sum + 1e-7) if e_sum > 0 else ent
    
    out = np.concatenate([hist_norm, ent_norm]).astype(np.float32)
    assert not np.isnan(out).any() and not np.isinf(out).any()
    return out

def extract_modality_b_tokens(record):
    """Modality B: Tokenized API imports, exports, and section names."""
    tokens = []
    imports = record.get('imports', {})
    if isinstance(imports, dict):
        for dll, funcs in imports.items():
            dll_clean = str(dll).lower().replace('.dll', '').strip()
            if dll_clean:
                tokens.append(f'dll_{dll_clean}')
            if isinstance(funcs, list):
                for fn in funcs:
                    fn_s = str(fn).lower().strip()
                    if fn_s:
                        tokens.append(f'fn_{fn_s}')
    exports = record.get('exports', [])
    if isinstance(exports, list):
        for exp in exports:
            exp_s = str(exp).lower().strip()
            if exp_s:
                tokens.append(f'exp_{exp_s}')
    sec_obj = record.get('section', {})
    if isinstance(sec_obj, dict):
        entry = sec_obj.get('entry', '')
        if entry:
            tokens.append(f'entry_{str(entry).lower().strip()}')
        for sec in sec_obj.get('sections', []):
            if isinstance(sec, dict) and sec.get('name'):
                s_name = str(sec['name']).lower().strip().replace('.', '')
                tokens.append(f'sec_{s_name}')
    return tokens

def extract_modality_c(record):
    """Modality C: 66-D Structural PE numerical features (log1p transformed)."""
    vec = []
    gen = record.get('general', {}) if isinstance(record.get('general'), dict) else {}
    vec.extend([
        math.log1p(max(0, float(gen.get('size', 0)))),
        math.log1p(max(0, float(gen.get('vsize', 0)))),
        float(gen.get('has_debug', 0)),
        math.log1p(max(0, float(gen.get('exports', 0)))),
        math.log1p(max(0, float(gen.get('imports', 0)))),
        float(gen.get('has_relocations', 0)),
        float(gen.get('has_resources', 0)),
        float(gen.get('has_signature', 0)),
        float(gen.get('has_tls', 0)),
        math.log1p(max(0, float(gen.get('symbols', 0))))
    ])
    
    hdr = record.get('header', {}) if isinstance(record.get('header'), dict) else {}
    coff = hdr.get('coff', {}) if isinstance(hdr.get('coff'), dict) else {}
    chars = coff.get('characteristics', [])
    vec.append(float(len(chars) if isinstance(chars, list) else 0))
    machine = str(coff.get('machine', '')).upper()
    vec.append(1.0 if ('386' in machine or 'I386' in machine) else 0.0)
    vec.append(1.0 if ('AMD64' in machine or '64' in machine) else 0.0)
    
    opt = hdr.get('optional', {}) if isinstance(hdr.get('optional'), dict) else {}
    subsys = str(opt.get('subsystem', '')).upper()
    vec.append(1.0 if 'GUI' in subsys else 0.0)
    vec.append(1.0 if 'CUI' in subsys else 0.0)
    vec.extend([
        float(opt.get('major_image_version', 0)),
        float(opt.get('minor_image_version', 0)),
        float(opt.get('major_linker_version', 0)),
        math.log1p(max(0, float(opt.get('sizeof_code', 0)))),
        math.log1p(max(0, float(opt.get('sizeof_headers', 0)))),
        math.log1p(max(0, float(opt.get('sizeof_heap_commit', 0))))
    ])
    
    sec_obj = record.get('section', {}) if isinstance(record.get('section'), dict) else {}
    sections = sec_obj.get('sections', [])
    if isinstance(sections, list) and len(sections) > 0:
        entropies = [float(s.get('entropy', 0)) for s in sections if isinstance(s, dict)]
        raw_sizes = [float(s.get('size', 0)) for s in sections if isinstance(s, dict)]
        vsizes = [float(s.get('vsize', 0)) for s in sections if isinstance(s, dict)]
        n_exec = sum(1 for s in sections if isinstance(s, dict) and any('EXECUTE' in str(p) for p in s.get('props', [])))
        n_write = sum(1 for s in sections if isinstance(s, dict) and any('WRITE' in str(p) for p in s.get('props', [])))
        vec.extend([
            float(len(sections)),
            float(np.mean(entropies)) if entropies else 0.0,
            float(np.max(entropies)) if entropies else 0.0,
            math.log1p(max(0, sum(raw_sizes))),
            math.log1p(max(0, sum(vsizes))),
            float(n_exec),
            float(n_write)
        ])
    else:
        vec.extend([0.0] * 7)
        
    datadirs = record.get('datadirectories', [])
    for i in range(15):
        if isinstance(datadirs, list) and i < len(datadirs) and isinstance(datadirs[i], dict):
            sz = float(datadirs[i].get('size', 0))
            vec.extend([math.log1p(max(0, sz)), 1.0 if sz > 0 else 0.0])
        else:
            vec.extend([0.0, 0.0])
            
    strings = record.get('strings', {}) if isinstance(record.get('strings'), dict) else {}
    vec.extend([
        math.log1p(max(0, float(strings.get('numstrings', 0)))),
        float(strings.get('avlength', 0)),
        math.log1p(max(0, float(strings.get('printables', 0)))),
        float(strings.get('entropy', 0)),
        math.log1p(max(0, float(strings.get('paths', 0)))),
        math.log1p(max(0, float(strings.get('urls', 0)))),
        math.log1p(max(0, float(strings.get('registry', 0)))),
        math.log1p(max(0, float(strings.get('MZ', 0))))
    ])
    
    assert len(vec) == 66, f'Expected 66 structural features, got {len(vec)}'
    out = np.array(vec, dtype=np.float32)
    assert not np.isnan(out).any() and not np.isinf(out).any()
    return out

## 4. Extract Proportional Shard-Balanced Splits & Tabular Features
- **Training Set:** 16,000 samples (8,000 benign, 8,000 malware)
- **Validation Set:** 4,000 samples (2,000 benign, 2,000 malware)
- Drawn proportionally across all 6 training shards (`train_features_0.jsonl` through `train_features_5.jsonl`).
- Unlabeled records (`label == -1`) are strictly excluded.
- Official 2,381 EMBER features extracted for tabular benchmark.

In [ ]:
N_TRAIN = 16000
N_VAL = 4000
TOTAL_RECS = N_TRAIN + N_VAL
TARGET_BENIGN = TOTAL_RECS // 2
TARGET_MALWARE = TOTAL_RECS // 2

# Proportional quotas derived from full-dataset audit
shard_b_avail = [50000, 52338, 46563, 47004, 48433, 55662]
shard_m_avail = [0, 63713, 47044, 47309, 48413, 93521]
tot_b = sum(shard_b_avail)
tot_m = sum(shard_m_avail)

b_quotas = [int(round(TARGET_BENIGN * (c / tot_b))) for c in shard_b_avail]
b_quotas[-1] += TARGET_BENIGN - sum(b_quotas)

m_quotas = [int(round(TARGET_MALWARE * (c / tot_m))) for c in shard_m_avail]
m_quotas[-1] += TARGET_MALWARE - sum(m_quotas)

print('Proportional Quota Allocation Across All 6 Shards:')
for i in range(6):
    print(f'  Shard train_features_{i}.jsonl -> Benign: {b_quotas[i]:>5,}, Malware: {m_quotas[i]:>5,}')

def reservoir_sample_shard(fpath, target_b, target_m, seed):
    rng = random.Random(seed)
    res_b, res_m = [], []
    count_b, count_m = 0, 0
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                rec = json.loads(line)
                lbl = rec.get('label')
                if lbl == 0:
                    count_b += 1
                    if len(res_b) < target_b:
                        res_b.append(rec)
                    else:
                        j = rng.randint(0, count_b - 1)
                        if j < target_b:
                            res_b[j] = rec
                elif lbl == 1:
                    count_m += 1
                    if len(res_m) < target_m:
                        res_m.append(rec)
                    else:
                        j = rng.randint(0, count_m - 1)
                        if j < target_m:
                            res_m[j] = rec
            except json.JSONDecodeError:
                continue
    return res_b, res_m

collected_b, collected_m = [], []
t0_load = time.time()
for i in range(6):
    fpath = os.path.join(DATA_DIR, f'train_features_{i}.jsonl')
    t_b, t_m = b_quotas[i], m_quotas[i]
    print(f'Streaming {os.path.basename(fpath)} via Reservoir Sampling (quota: {t_b} B, {t_m} M)...')
    s_b, s_m = reservoir_sample_shard(fpath, t_b, t_m, seed=SEED + i * 17)
    collected_b.extend(s_b)
    collected_m.extend(s_m)
    print(f'  -> Shard {i} complete: collected {len(s_b)} B, {len(s_m)} M')

print(f'Loaded {len(collected_b):,} Benign and {len(collected_m):,} Malware in {time.time()-t0_load:.1f} s.')

# Stratified Partitioning
rng = random.Random(SEED)
rng.shuffle(collected_b)
rng.shuffle(collected_m)

train_recs = collected_b[:N_TRAIN//2] + collected_m[:N_TRAIN//2]
val_recs = collected_b[N_TRAIN//2:N_TRAIN//2 + N_VAL//2] + collected_m[N_TRAIN//2:N_TRAIN//2 + N_VAL//2]
rng.shuffle(train_recs)
rng.shuffle(val_recs)

train_shas = [r['sha256'] for r in train_recs]
val_shas = [r['sha256'] for r in val_recs]
overlap = set(train_shas).intersection(set(val_shas))
print(f'Train / Val SHA-256 Hash Overlap: {len(overlap)} (MUST BE ZERO)')
assert len(overlap) == 0, 'Data leakage: Train and validation sets overlap!'

y_train = np.array([int(r['label']) for r in train_recs], dtype=np.int64)
y_val = np.array([int(r['label']) for r in val_recs], dtype=np.int64)

# 1. Modality A (Byte statistics)
print('\nExtracting Modality A (512-D Byte Statistics)...')
X_byte_train = np.stack([extract_modality_a(r) for r in train_recs], axis=0)
X_byte_val = np.stack([extract_modality_a(r) for r in val_recs], axis=0)

# 2. Modality B (Vocabulary fitted ONLY on train split)
print('Building vocabulary strictly from 16k training split (Modality B)...')
train_token_lists = [extract_modality_b_tokens(r) for r in train_recs]
val_token_lists = [extract_modality_b_tokens(r) for r in val_recs]

token_counts = Counter(t for sub in train_token_lists for t in sub)
vocab = {'<PAD>': 0, '<UNK>': 1}
for tok, _ in token_counts.most_common(VOCAB_SIZE - 2):
    vocab[tok] = len(vocab)
print(f'[OK] Vocabulary size: {len(vocab):,} tokens.')

def encode_seq(tokens, vocab, max_len=MAX_SEQ_LEN):
    ids = [vocab.get(t, 1) for t in tokens[:max_len]]
    if len(ids) < max_len:
        ids.extend([0] * (max_len - len(ids)))
    return ids

X_text_train = np.array([encode_seq(t, vocab) for t in train_token_lists], dtype=np.int64)
X_text_val = np.array([encode_seq(t, vocab) for t in val_token_lists], dtype=np.int64)

# 3. Modality C (StandardScaler fitted ONLY on train split)
print('Extracting Modality C (66-D Structural PE Metrics)...')
X_struct_raw_train = np.stack([extract_modality_c(r) for r in train_recs], axis=0)
X_struct_raw_val = np.stack([extract_modality_c(r) for r in val_recs], axis=0)

scaler = StandardScaler()
X_struct_train = scaler.fit_transform(X_struct_raw_train).astype(np.float32)
X_struct_val = scaler.transform(X_struct_raw_val).astype(np.float32)

# 4. Official EMBER 2,381 Features (for Tabular Baseline)
print('Extracting official EMBER 2,381-dimensional features...')
ember_extractor = ember.features.PEFeatureExtractor(feature_version=2)
X_ember_train = np.array([ember_extractor.process_raw_features(r) for r in train_recs], dtype=np.float32)
X_ember_val = np.array([ember_extractor.process_raw_features(r) for r in val_recs], dtype=np.float32)

    # Cache pre-extracted arrays for instant future loading
    np.savez_compressed(
        os.path.join(DATA_CACHE_DIR, 'ember_train_16k_corrected.npz'),
        y=y_train,
        X_byte=X_byte_train,
        X_text=X_text_train,
        X_struct=X_struct_train,
        X_ember2381=X_ember_train,
        shas=np.array(train_shas)
    )
    np.savez_compressed(
        os.path.join(DATA_CACHE_DIR, 'ember_val_4k_corrected.npz'),
        y=y_val,
        X_byte=X_byte_val,
        X_text=X_text_val,
        X_struct=X_struct_val,
        X_ember2381=X_ember_val,
        shas=np.array(val_shas)
    )
    with open(os.path.join(DATA_CACHE_DIR, 'vocab.json'), 'w', encoding='utf-8') as f:
        json.dump(vocab, f, indent=2)
    print(f'[OK] Cached pre-extracted dataset arrays in: {DATA_CACHE_DIR}')

assert X_ember_train.shape == (16000, 2381), f'Unexpected train shape: {X_ember_train.shape}'
assert X_ember_val.shape == (4000, 2381), f'Unexpected val shape: {X_ember_val.shape}'
assert np.isfinite(X_ember_train).all(), 'NaN/Inf in X_ember_train'
assert np.isfinite(X_ember_val).all(), 'NaN/Inf in X_ember_val'

print('[OK] All five feature matrices prepared and verified finite with zero data leakage.')

## 5. Architectural Verification: Padding Invariance & Gradient Flow Tests
- **Test 1:** MaskedTextCNN padding invariance under variable length padding (10, 50, 256).
- **Test 2:** End-to-end multimodal gradient flow asserting active gradients across all 3 encoders and classifier.

In [ ]:
# 1. ByteMLP (Modality A: 512-D)
class ByteMLP(nn.Module):
    def __init__(self, in_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x_byte):
        return self.net(x_byte)

# 2. MaskedTextCNN (Modality B: Padding-Safe with Masked Average Pooling)
class MaskedTextCNN(nn.Module):
    def __init__(self, vocab_sz=10000, emb_dim=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_sz, emb_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(emb_dim, 128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.clf = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x_text):
        mask = (x_text != 0).float() # (batch, seq_len)
        h = self.emb(x_text).transpose(1, 2) # (batch, emb_dim, seq_len)
        
        # Apply mask after convolutions
        h1 = self.relu(self.conv1(h)) * mask.unsqueeze(1)
        h2 = self.relu(self.conv2(h1)) * mask.unsqueeze(1)
        
        # Masked Average Pooling (divides strictly by real token count)
        valid_counts = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        pooled = h2.sum(dim=-1) / valid_counts
        return self.clf(pooled)

# 3. StructMLP (Modality C: 66-D)
class StructMLP(nn.Module):
    def __init__(self, in_dim=66):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x_struct):
        return self.net(x_struct)

# 4. MultimodalFusionNet (A + B + C Joint Multimodal Baseline)
class MultimodalFusionNet(nn.Module):
    def __init__(self, vocab_sz=10000, byte_dim=512, struct_dim=66, text_emb_dim=128):
        super().__init__()
        self.byte_enc = nn.Sequential(nn.Linear(byte_dim, 128), nn.BatchNorm1d(128), nn.ReLU())
        self.struct_enc = nn.Sequential(nn.Linear(struct_dim, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.text_emb = nn.Embedding(vocab_sz, text_emb_dim, padding_idx=0)
        self.text_conv1 = nn.Conv1d(text_emb_dim, 128, kernel_size=3, padding=1)
        self.text_conv2 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        
        # Joint Concatenation: 128 (byte) + 128 (text) + 64 (struct) = 320
        self.classifier = nn.Sequential(
            nn.Linear(320, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x_byte, x_text, x_struct):
        h_b = self.byte_enc(x_byte)
        h_s = self.struct_enc(x_struct)
        
        mask = (x_text != 0).float()
        t_h = self.text_emb(x_text).transpose(1, 2)
        t_h1 = self.relu(self.text_conv1(t_h)) * mask.unsqueeze(1)
        t_h2 = self.relu(self.text_conv2(t_h1)) * mask.unsqueeze(1)
        valid_counts = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        h_t = t_h2.sum(dim=-1) / valid_counts
        
        fused = torch.cat([h_b, h_t, h_s], dim=1) # (batch, 320)
        return self.classifier(fused)

# Run Unit Tests
print('=' * 65)
print('UNIT TEST 1: Padding Invariance in MaskedTextCNN')
print('=' * 65)
test_cnn = MaskedTextCNN(vocab_sz=len(vocab)).to(device)
test_cnn.eval()
seq_5 = torch.tensor([[10, 25, 42, 99, 7]], device=device)
seq_10 = torch.tensor([[10, 25, 42, 99, 7] + [0]*5], device=device)
seq_50 = torch.tensor([[10, 25, 42, 99, 7] + [0]*45], device=device)
seq_256 = torch.tensor([[10, 25, 42, 99, 7] + [0]*251], device=device)
with torch.no_grad():
    d_10 = torch.max(torch.abs(test_cnn(seq_5) - test_cnn(seq_10))).item()
    d_50 = torch.max(torch.abs(test_cnn(seq_5) - test_cnn(seq_50))).item()
    d_256 = torch.max(torch.abs(test_cnn(seq_5) - test_cnn(seq_256))).item()
print(f'  Max logit diff (5 vs 10 tokens) : {d_10:.8e}')
print(f'  Max logit diff (5 vs 50 tokens) : {d_50:.8e}')
print(f'  Max logit diff (5 vs 256 tokens): {d_256:.8e}')
assert d_256 < 1e-6, 'Padding leakage detected!'
print('  [PASS] MaskedTextCNN is mathematically invariant to padding length.\n')

print('=' * 65)
print('UNIT TEST 2: Multimodal Fusion Gradient Flow')
print('=' * 65)
test_mm = MultimodalFusionNet(vocab_sz=len(vocab)).to(device)
test_mm.train()
d_b = torch.randn(4, 512, device=device)
d_t = torch.randint(0, len(vocab), (4, 64), device=device)
d_s = torch.randn(4, 66, device=device)
d_y = torch.tensor([0, 1, 0, 1], device=device)
crit = nn.CrossEntropyLoss()
opt = optim.Adam(test_mm.parameters(), lr=1e-3)
opt.zero_grad()
out = test_mm(d_b, d_t, d_s)
loss = crit(out, d_y)
loss.backward()

g_b = test_mm.byte_enc[0].weight.grad
g_t_emb = test_mm.text_emb.weight.grad
g_t_conv = test_mm.text_conv1.weight.grad
g_s = test_mm.struct_enc[0].weight.grad
g_clf = test_mm.classifier[0].weight.grad

print(f'  Byte Encoder Grad Norm   : {g_b.norm().item():.6f}')
print(f'  Text Embedding Grad Norm : {g_t_emb.norm().item():.6f}')
print(f'  Text Conv1 Grad Norm     : {g_t_conv.norm().item():.6f}')
print(f'  Struct Encoder Grad Norm : {g_s.norm().item():.6f}')
print(f'  Fusion Clf Grad Norm     : {g_clf.norm().item():.6f}')

assert g_b.norm().item() > 0, 'Byte encoder received zero gradient!'
assert g_t_conv.norm().item() > 0, 'Text encoder received zero gradient!'
assert g_s.norm().item() > 0, 'Struct encoder received zero gradient!'
assert g_clf.norm().item() > 0, 'Classifier received zero gradient!'
assert not torch.isnan(out).any() and not torch.isinf(out).any(), 'NaN/Inf in model output!'
print('  [PASS] All 3 modality encoders receive active backpropagated gradients.')
print('=' * 65)

## 6. Experiment 1: Fresh LightGBM Baseline (Fair Tabular Benchmark)
- **Training Data:** Strictly the 16,000 training records (`X_ember_train`)
- **Validation Data:** Strictly the 4,000 validation records (`X_ember_val`)
- **Configuration:** `n_estimators=1000`, `learning_rate=0.05`, `num_leaves=31`, `random_state=42`
- **Feature Space:** Official 2,381 EMBER features

In [ ]:
print('=' * 65)
print('EXPERIMENT 1: FRESH LIGHTGBM TABULAR BASELINE')
print('=' * 65)

# Instantiate fresh classifier with audited hyperparameters
fresh_lgb = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)

print('Fitting Fresh LightGBM on 16,000 training samples...')
t0_lgb = time.time()
fresh_lgb.fit(X_ember_train, y_train)
fresh_tr_time = round(time.time() - t0_lgb, 2)
print(f'[OK] Training completed in {fresh_tr_time} s.')

# Validation inference & latency
t0_eval = time.time()
fresh_probs = fresh_lgb.predict_proba(X_ember_val)[:, 1]
fresh_latency = (time.time() - t0_eval) / len(y_val) * 1000
fresh_preds = (fresh_probs >= 0.5).astype(int)

# Compute metrics
fresh_acc = float(accuracy_score(y_val, fresh_preds))
fresh_prec = float(precision_score(y_val, fresh_preds))
fresh_rec = float(recall_score(y_val, fresh_preds))
fresh_f1 = float(f1_score(y_val, fresh_preds))
fresh_auc = float(roc_auc_score(y_val, fresh_probs))
fresh_cm = confusion_matrix(y_val, fresh_preds)
tn, fp, fn, tp = fresh_cm.ravel()
fresh_fpr = float(fp / max(1, (fp + tn)))

# FPR at 99% TPR
fpr_arr, tpr_arr, _ = roc_curve(y_val, fresh_probs)
idx_99 = np.where(tpr_arr >= 0.99)[0]
fresh_fpr_99 = float(fpr_arr[idx_99[0]]) if len(idx_99) > 0 else 1.0

print('\n--- Fresh LightGBM Validation Results ---')
print(f'  Accuracy         : {fresh_acc:.4f}')
print(f'  Precision        : {fresh_prec:.4f}')
print(f'  Recall           : {fresh_rec:.4f}')
print(f'  F1-Score         : {fresh_f1:.4f}')
print(f'  ROC-AUC          : {fresh_auc:.4f}')
print(f'  False Pos Rate   : {fresh_fpr:.4f}')
print(f'  FPR @ 99% TPR    : {fresh_fpr_99:.4f}')
print(f'  Malware FN Misses: {fn:,} (out of {tp+fn:,} malware)')
print(f'  Training Time    : {fresh_tr_time} s')
print(f'  Inference Latency: {fresh_latency:.3f} ms/sample')

# Save model checkpoint and confusion matrix plot
lgb_model_path = os.path.join(CHECKPOINTS_DIR, 'fresh_lightgbm.txt')
fresh_lgb.booster_.save_model(lgb_model_path)
print(f'Saved checkpoint: {lgb_model_path}')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(fresh_cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Benign (0)', 'Malware (1)'])
ax.set_yticklabels(['Benign (0)', 'Malware (1)'])
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Fresh LightGBM Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{fresh_cm[i, j]:,}', ha='center', va='center', color='red' if (i==1 and j==0) else 'black')
plt.tight_layout()
cm_path = os.path.join(PLOTS_DIR, 'cm_fresh_lightgbm.png')
plt.savefig(cm_path, dpi=150)
plt.close()

res_lgb = {
    'Model': 'Experiment 1: Fresh LightGBM', 
    'Modality': 'Tabular (2,381 Features)',
    'Accuracy': round(fresh_acc, 4),
    'Precision': round(fresh_prec, 4),
    'Recall': round(fresh_rec, 4),
    'F1-Score': round(fresh_f1, 4),
    'ROC-AUC': round(fresh_auc, 4),
    'FPR': round(fresh_fpr, 4),
    'FPR@99%TPR': round(fresh_fpr_99, 4),
    'Malware_FN': int(fn),
    'Training_Time_s': fresh_tr_time,
    'Latency_ms': round(fresh_latency, 3)
}

## 6B. Pretrained LightGBM Reference Documentation (DO NOT EXECUTE AS FAIR BASELINE)

> [!CAUTION]
> **Methodological Warning:** `ember_model_2018.txt` was trained on the **entire 600,000 labeled EMBER 2018 training population** (`internal_count=600000`). Because our 4,000 validation records are drawn from `train_features_*.jsonl`, **every validation sample was present in its training set**. Evaluating it here represents **in-sample memorization**, NOT out-of-sample generalization. Per Phase 2 instructions, this cell is retained solely as documentation and is NOT executed or included in the fair baseline comparison.

In [ ]:
# ==============================================================================
# IN-SAMPLE REFERENCE ONLY - INACTIVE / DO NOT EXECUTE FOR FAIR BENCHMARKING
# ==============================================================================
"""
# To verify the in-sample literature reference (~92.5% target):
pre_model_path = os.path.join(DATA_DIR, 'ember_model_2018.txt')
if os.path.exists(pre_model_path):
    pre_booster = lgb.Booster(model_file=pre_model_path)
    pre_probs = pre_booster.predict(X_ember_val)
    print('Pretrained In-Sample AUC:', roc_auc_score(y_val, pre_probs))
"""
print('[INFO] Pretrained LightGBM evaluation cell is quarantined and bypassed to prevent in-sample leakage.')

## 7. Train & Evaluate Four Controlled Neural Baselines
All four neural models use identical training configurations:
- **Optimizer:** Adam (`lr=0.001`, `weight_decay=0.0001`)
- **Batch Size:** Adapted to runtime (`BATCH_SIZE` = 64 on GPU, 32 on CPU)
- **Loss Function:** `nn.CrossEntropyLoss()`
- **Epochs:** 10 with best validation loss checkpoint tracking

In [ ]:
def train_and_evaluate_neural_model(model_name, modality_label, model, train_loader, val_loader, epochs=10, lr=1e-3):
    print('=' * 65)
    print(f'TRAINING: {model_name}')
    print('=' * 65)
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    
    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_ckpt_path = os.path.join(CHECKPOINTS_DIR, f'best_{model_name.replace(" ", "_").replace(":", "")}.pt')
    
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        total_tr_loss = 0.0
        for batch in train_loader:
            inputs = [item.to(device) for item in batch[:-1]]
            y_b = batch[-1].to(device)
            
            opt.zero_grad()
            out = model(*inputs)
            loss = crit(out, y_b)
            loss.backward()
            opt.step()
            total_tr_loss += loss.item() * len(y_b)
            
        ep_tr_loss = total_tr_loss / len(train_loader.dataset)
        train_losses.append(ep_tr_loss)
        
        # Validation Loss Check
        model.eval()
        total_va_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs = [item.to(device) for item in batch[:-1]]
                y_b = batch[-1].to(device)
                v_loss = crit(model(*inputs), y_b)
                total_va_loss += v_loss.item() * len(y_b)
        ep_va_loss = total_va_loss / len(val_loader.dataset)
        val_losses.append(ep_va_loss)
        
        if ep_va_loss < best_val_loss:
            best_val_loss = ep_va_loss
            torch.save(model.state_dict(), best_ckpt_path)
            
        print(f'  Epoch {epoch:>2}/{epochs} -> Train Loss: {ep_tr_loss:.4f} | Val Loss: {ep_va_loss:.4f} | Best: {best_val_loss:.4f}')
        
    tr_duration = round(time.time() - t0, 2)
    print(f'[OK] Training finished in {tr_duration} s. Best checkpoint saved.')
    
    # Load best checkpoint for validation evaluation
    model.load_state_dict(torch.load(best_ckpt_path))
    model.eval()
    
    val_probs = []
    t0_eval = time.time()
    with torch.no_grad():
        for batch in val_loader:
            inputs = [item.to(device) for item in batch[:-1]]
            logits = model(*inputs)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            val_probs.extend(probs)
    val_latency = (time.time() - t0_eval) / len(val_loader.dataset) * 1000
    val_probs = np.array(val_probs, dtype=np.float32)
    preds = (val_probs >= 0.5).astype(int)
    
    # Save predictions and probabilities
    pred_file = os.path.join(METRICS_DIR, f'preds_{model_name.replace(" ", "_").replace(":", "")}.npz')
    np.savez_compressed(pred_file, y_true=y_val, y_probs=val_probs, y_pred=preds)
    
    # Compute all metrics
    acc = float(accuracy_score(y_val, preds))
    prec = float(precision_score(y_val, preds))
    rec = float(recall_score(y_val, preds))
    f1 = float(f1_score(y_val, preds))
    auc = float(roc_auc_score(y_val, val_probs))
    cm = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm.ravel()
    fpr = float(fp / max(1, (fp + tn)))
    
    fpr_arr, tpr_arr, _ = roc_curve(y_val, val_probs)
    idx_99 = np.where(tpr_arr >= 0.99)[0]
    fpr_99 = float(fpr_arr[idx_99[0]]) if len(idx_99) > 0 else 1.0
    
    print(f'\n--- {model_name} Final Validation Results ---')
    print(f'  Accuracy         : {acc:.4f}')
    print(f'  Precision        : {prec:.4f}')
    print(f'  Recall           : {rec:.4f}')
    print(f'  F1-Score         : {f1:.4f}')
    print(f'  ROC-AUC          : {auc:.4f}')
    print(f'  False Pos Rate   : {fpr:.4f}')
    print(f'  FPR @ 99% TPR    : {fpr_99:.4f}')
    print(f'  Malware FN Misses: {fn:,} (out of {tp+fn:,} malware)')
    print(f'  Inference Latency: {val_latency:.3f} ms/sample\n')
    
    # Plot and save Loss Curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(range(1, epochs + 1), train_losses, label='Train Loss', color='blue')
    ax1.plot(range(1, epochs + 1), val_losses, label='Val Loss', color='orange')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{model_name} Loss Curve')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot and save Confusion Matrix
    ax2.imshow(cm, cmap='Blues')
    ax2.set_xticks([0, 1])
    ax2.set_yticks([0, 1])
    ax2.set_xticklabels(['Benign', 'Malware'])
    ax2.set_yticklabels(['Benign', 'Malware'])
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('True')
    ax2.set_title(f'{model_name} Confusion Matrix')
    for i in range(2):
        for j in range(2):
            ax2.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', color='red' if (i==1 and j==0) else 'black')
    plt.tight_layout()
    plot_path = os.path.join(PLOTS_DIR, f'plot_{model_name.replace(" ", "_").replace(":", "")}.png')
    plt.savefig(plot_path, dpi=150)
    plt.close()
    
    return {
        'Model': model_name,
        'Modality': modality_label,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4),
        'FPR': round(fpr, 4),
        'FPR@99%TPR': round(fpr_99, 4),
        'Malware_FN': int(fn),
        'Training_Time_s': tr_duration,
        'Latency_ms': round(val_latency, 3),
        'val_probs': val_probs
    }

# Prepare PyTorch Tensor DataLoaders
t_b_tr = torch.tensor(X_byte_train, dtype=torch.float32)
t_b_va = torch.tensor(X_byte_val, dtype=torch.float32)
t_t_tr = torch.tensor(X_text_train, dtype=torch.long)
t_t_va = torch.tensor(X_text_val, dtype=torch.long)
t_s_tr = torch.tensor(X_struct_train, dtype=torch.float32)
t_s_va = torch.tensor(X_struct_val, dtype=torch.float32)
t_y_tr = torch.tensor(y_train, dtype=torch.long)
t_y_va = torch.tensor(y_val, dtype=torch.long)

# 1. Byte-only Loader
ld_byte_tr = DataLoader(TensorDataset(t_b_tr, t_y_tr), batch_size=BATCH_SIZE, shuffle=True)
ld_byte_va = DataLoader(TensorDataset(t_b_va, t_y_va), batch_size=BATCH_SIZE, shuffle=False)

# 2. Text-only Loader
ld_text_tr = DataLoader(TensorDataset(t_t_tr, t_y_tr), batch_size=BATCH_SIZE, shuffle=True)
ld_text_va = DataLoader(TensorDataset(t_t_va, t_y_va), batch_size=BATCH_SIZE, shuffle=False)

# 3. Struct-only Loader
ld_struct_tr = DataLoader(TensorDataset(t_s_tr, t_y_tr), batch_size=BATCH_SIZE, shuffle=True)
ld_struct_va = DataLoader(TensorDataset(t_s_va, t_y_va), batch_size=BATCH_SIZE, shuffle=False)

# 4. Multimodal Loader
ld_mm_tr = DataLoader(TensorDataset(t_b_tr, t_t_tr, t_s_tr, t_y_tr), batch_size=BATCH_SIZE, shuffle=True)
ld_mm_va = DataLoader(TensorDataset(t_b_va, t_t_va, t_s_va, t_y_va), batch_size=BATCH_SIZE, shuffle=False)

# Run Controlled Experiments
res_byte = train_and_evaluate_neural_model('Experiment 2: ByteMLP', 'Modality A (Byte 512-D)', ByteMLP(), ld_byte_tr, ld_byte_va)
res_text = train_and_evaluate_neural_model('Experiment 3: MaskedTextCNN', 'Modality B (Text Seq 256)', MaskedTextCNN(vocab_sz=len(vocab)), ld_text_tr, ld_text_va)
res_struct = train_and_evaluate_neural_model('Experiment 4: StructMLP', 'Modality C (Struct 66-D)', StructMLP(), ld_struct_tr, ld_struct_va)
res_mm = train_and_evaluate_neural_model('Experiment 5: MultimodalFusionNet', 'Modality A+B+C (Joint 320-D)', MultimodalFusionNet(vocab_sz=len(vocab)), ld_mm_tr, ld_mm_va)

## 8. Consolidated Results Analysis, Visualizations & Manifest Export

In [ ]:
# Compile Fair 5-Baseline Comparison
all_results = [
    res_lgb,
    {k: v for k, v in res_byte.items() if k != 'val_probs'},
    {k: v for k, v in res_text.items() if k != 'val_probs'},
    {k: v for k, v in res_struct.items() if k != 'val_probs'},
    {k: v for k, v in res_mm.items() if k != 'val_probs'}
]

df_comparison = pd.DataFrame(all_results)
print('=' * 85)
print('FAIR BASELINE COMPARISON TABLE (All Models: 16,000 Train -> 4,000 Val)')
print('=' * 85)
display(df_comparison)

# 1. Plot Comparative Bar Chart
fig, ax = plt.subplots(figsize=(10, 5))
models_list = [r['Model'].split(': ')[1] for r in all_results]
x = np.arange(len(models_list))
w = 0.25

ax.bar(x - w, [r['Accuracy'] for r in all_results], width=w, label='Accuracy', color='#2b5c8f')
ax.bar(x, [r['F1-Score'] for r in all_results], width=w, label='F1-Score', color='#d95f02')
ax.bar(x + w, [r['ROC-AUC'] for r in all_results], width=w, label='ROC-AUC', color='#7570b3')

ax.set_xticks(x)
ax.set_xticklabels(models_list, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim([0.7, 1.0])
ax.set_title('Controlled Baseline Comparison (16,000 Train / 4,000 Validation)')
ax.legend(loc='lower right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
bar_chart_path = os.path.join(PLOTS_DIR, 'baseline_metrics_comparison.png')
plt.savefig(bar_chart_path, dpi=150)
plt.close()

# 2. Plot Combined ROC Curves
plt.figure(figsize=(8, 6))
probs_dict = {
    'Fresh LightGBM': fresh_probs,
    'ByteMLP': res_byte['val_probs'],
    'MaskedTextCNN': res_text['val_probs'],
    'StructMLP': res_struct['val_probs'],
    'MultimodalFusionNet': res_mm['val_probs']
}
for m_label, p_arr in probs_dict.items():
    fpr_c, tpr_c, _ = roc_curve(y_val, p_arr)
    auc_c = roc_auc_score(y_val, p_arr)
    plt.plot(fpr_c, tpr_c, label=f'{m_label} (AUC = {auc_c:.4f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Chance')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Validation ROC Curves Across All 5 Baselines')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
roc_path = os.path.join(PLOTS_DIR, 'combined_roc_curves.png')
plt.savefig(roc_path, dpi=150)
plt.close()

# Save Consolidated Metrics JSON
out_metrics_file = os.path.join(METRICS_DIR, 'baseline_correction_results.json')
with open(out_metrics_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2)
print(f'[OK] Saved consolidated metrics to: {out_metrics_file}')

# Save Complete Experiment Reproducibility Manifest
manifest = {
    'experiment_name': 'ember_native_multimodal_baseline_correction',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()),
    'random_seed': SEED,
    'dataset_info': {
        'source': 'EMBER 2018 v2',
        'train_shards_used': [f'train_features_{i}.jsonl' for i in range(6)],
        'test_set_status': 'Strictly Quarantined (test_features.jsonl)',
        'n_train': N_TRAIN,
        'n_val': N_VAL,
        'train_hashes_sample': train_shas[:10] if 'train_shas' in locals() else [],
        'val_hashes_sample': val_shas[:10] if 'val_shas' in locals() else [],
        'train_val_hash_overlap': len(overlap) if 'overlap' in locals() else 0
    },
    'runtime_environment': {
        'python_version': sys.version.split()[0],
        'pytorch_version': torch.__version__,
        'lightgbm_version': lgb.__version__,
        'device': str(device),
        'gpu_model': gpu_name,
        'vram_gb': round(gpu_vram, 2),
        'batch_size': BATCH_SIZE
    },
    'hyperparameters': {
        'fresh_lightgbm': {'n_estimators': 1000, 'learning_rate': 0.05, 'num_leaves': 31, 'seed': SEED},
        'neural_models': {'optimizer': 'Adam', 'lr': 0.001, 'weight_decay': 0.0001, 'epochs': 10, 'batch_size': BATCH_SIZE},
        'text_representation': {'max_seq_len': MAX_SEQ_LEN, 'vocab_size': len(vocab), 'pooling': 'MaskedAveragePooling'}
    },
    'results': all_results
}
manifest_path = os.path.join(CONFIGS_DIR, 'experiment_manifest.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)
print(f'[OK] Saved complete reproducibility manifest to: {manifest_path}')
print('\n' + '=' * 85)
print('ALL 5 CONTROLLED BASELINE EXPERIMENTS COMPLETED SUCCESSFULLY!')
print('Outputs saved to:', EXP_ROOT_DIR)
print('=' * 85)